# 4D SfM — full batch (all dates)

The production run: processes **every dated image set** found in the time-lapse
directory, across the entire record (multi-year capable — not limited to one
year). Same per-date workflow as `4d_sfm_dem_monthly.ipynb`, which instead
processes one hand-picked date per month.

For each date it produces DEM + orthoimage + DoD + stable-terrain DoD + M3C2
raster (with histograms). All logic lives in
`tlapse4d.pipeline_4dsfm`; this notebook only says *which glacier*, *which
knobs*, and *which dates*.

> **Resume:** the per-date pipeline caches every step (`overwrite=False`), so
> re-running this notebook skips already-finished dates cheaply — safe to stop
> and restart on a long run. A date that fails (e.g. cloud-cover gate) is
> recorded as an error row and the batch continues.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ["AGISOFT_LICENSE_PATH"] = "/home/asus/.config/Agisoft/license.lic"

import Metashape  # noqa: F401  — must import after AGISOFT_LICENSE_PATH is set
from tlapse4d.pipeline_4dsfm import run_batch, select_dates

## Configuration

Paths (*which glacier*) come from `site_config*.py`; pipeline knobs (*how it is
run*) come from `run_config.py`. Override a knob here rather than editing that
file — e.g. `params = params | dict(verbose=True)`.

In [ ]:
import site_config_north as site
from run_config import params

# Inclusive date bounds ("YYYY-MM-DD"); None either side means no bound.
date_from = "2024-07-15"
date_to   = None

## Run

`select_dates` scans the time-lapse directory, drops the reference/baseline
day(s) read from the registry, and applies the date bounds. `run_batch` then
runs each date, records failures instead of stopping, prints a summary and
writes `output/batch_summary.csv`.

In [ ]:
dates = select_dates(
    site.tlcam_dir, site.registry_csv,
    date_from       = date_from,
    date_to         = date_to,
    # Select on the same filters the pipeline runs with, so a date whose frames
    # are all filtered away is never selected and then failed on.
    time_window     = params["time_window"],
    exclude_cameras = params["exclude_cameras"],
)

df = run_batch(
    dates,
    tlcam_dir    = site.tlcam_dir,
    ref_cloud    = site.ref_cloud,
    glacier_mask = site.glacier_mask,
    registry_csv = site.registry_csv,
    output_dir   = site.output_dir,
    **params,
)
df